<a href="https://colab.research.google.com/github/codebysumit/cryptography-algorithms/blob/master/notebooks/columnar_transposition_multi_round.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Columnar Transposition Technique with Multiple Rounds

## History
A single round of Columnar Transposition is easy to break once an attacker suspects a transposition cipher is being used, because the letter frequencies of the plaintext are still sitting right there in the ciphertext, just rearranged. Cryptographers realised that running the ciphertext through a **second round** of columnar transposition, usually with a different keyword, scrambles the message far more thoroughly. This idea is called **Double Columnar Transposition**, and it was used seriously in both World Wars. The German ADFGVX cipher, used in World War 1, combined a substitution step with a double columnar transposition step, and it gave Allied codebreakers a genuinely hard time before it was finally broken.

## What is Multi-Round Columnar Transposition?
This notebook takes the single-round Columnar Transposition Cipher and simply **runs it multiple times in a row**, using a different keyword for each round. Each round is still exactly the same transposition process: fill a grid row by row, read it back column by column in keyword rank order.

The key idea is this: after round 1 scrambles the plaintext, round 2 does not see the original plaintext anymore, it scrambles the **already scrambled output of round 1**. This means the position of any single original character gets mixed twice, which spreads plaintext patterns out much further than a single round ever could.

This notebook supports any number of rounds, and each round can use a keyword of a different length, since transposition does not care what came before it, it only cares about the length of whatever text it currently has in front of it.

In this implementation, we use the printable ASCII range from space (` `) to tilde (`~`), which spans from ASCII value 32 to 126. Since this cipher only rearranges characters instead of transforming them, every character in that range is treated exactly the same way, in every round.

## Cryptography Algorithm

### Constants
*   $S = 32$ (Start of printable ASCII range)
*   $E = 126$ (End of printable ASCII range)
*   $K_1, K_2, ..., K_r$ = the keywords used for round 1, round 2, up to round $r$
*   $r$ = the number of rounds

### 1. Single Round Recap
Each round is the same Columnar Transposition process covered in the earlier notebook:
1.  Rank the letters of the round's keyword alphabetically to decide column reading order.
2.  Write the current text into a grid, row by row, using as many columns as the keyword has letters.
3.  Read the grid back out column by column, following the rank order, to produce this round's output.

### 2. Multi-Round Encryption
The output of round 1 becomes the input to round 2, and so on, all the way through round $r$:

$$C_1 = \text{Transpose}(P, K_1)$$
$$C_2 = \text{Transpose}(C_1, K_2)$$
$$\vdots$$
$$C_r = \text{Transpose}(C_{r-1}, K_r)$$

The final ciphertext is $C_r$, the output after the last round.

### 3. Multi-Round Decryption
Decryption must undo the rounds in the **exact opposite order** they were applied, using the matching inverse of each round:

$$C_{r-1} = \text{Untranspose}(C_r, K_r)$$
$$\vdots$$
$$P = \text{Untranspose}(C_1, K_1)$$

In other words, the **last** keyword used during encryption must be the **first** keyword used during decryption. Getting the round order backwards, even with all the correct keywords, will not recover the plaintext.

### 4. Fully Worked Example (By Hand, 2 Rounds)
Let's encrypt **ATTACKATDAWN** using **Round 1 keyword: HACK**, then **Round 2 keyword: ZEBRA**.

**Round 1** (same as the single round example from the earlier notebook):

```
Column:   H  A  C  K
Rank:     2  0  1  3

Row 1:    A  T  T  A
Row 2:    C  K  A  T
Row 3:    D  A  W  N
```

Reading columns in rank order (A, C, H, K): **TKA** + **TAW** + **ACD** + **ATN** = **TKATAWACDATN**

**Round 2** takes TKATAWACDATN as its new input, and uses keyword **ZEBRA** (5 columns, ranks Z=4, E=1, B=0, R=3, A=2 wait let's rank properly: alphabetically A < B < E < R < Z, so A=0, B=1, E=2, R=3, Z=4).

```
Column:   Z  E  B  R  A
Rank:     4  2  1  3  0

Row 1:    T  K  A  T  A
Row 2:    W  A  C  D  A
Row 3:    T  N  (empty)(empty)(empty)
```

This time the message length (12) does not divide evenly by 5 columns, so the last row is short, only columns Z and E get a character in row 3.

Reading columns in rank order (A rank0, B rank1, E rank2, R rank3, Z rank4): column A = **AAA**, column B = **AC**, column E = **KAN**, column R = **TD**, column Z = **TWT**

**Final Ciphertext = AAACKANTDTWT**

This is exactly what the code below computes automatically.

### 1. Import Dependencies

In [1]:
import random
import string
import math

### 2. Helper Utilities

In [2]:
START_ASCII = 32
END_ASCII = 126

def validate_printable_text(text: str) -> None:
    for ch in text:
        code = ord(ch)
        if not (START_ASCII <= code <= END_ASCII):
            raise ValueError(
                f"Character {ch!r} (ASCII {code}) is outside the supported range "
                f"{START_ASCII}-{END_ASCII}."
            )

def build_column_order(keyword: str) -> list:
    # rank[i] = the reading order rank of column i, based on alphabetical order of the keyword
    indexed = list(enumerate(keyword))
    sorted_indexed = sorted(indexed, key=lambda pair: (pair[1], pair[0]))

    order = [0] * len(keyword)
    for rank, (original_index, ch) in enumerate(sorted_indexed):
        order[original_index] = rank

    return order

### 3. Single Round Encryption and Decryption

In [3]:
def single_round_encrypt(text: str, keyword: str) -> str:
    validate_printable_text(text)

    num_cols = len(keyword)
    if num_cols < 2:
        raise ValueError("Each round's keyword must be at least 2 characters long.")

    order = build_column_order(keyword)
    length = len(text)

    num_rows = math.ceil(length / num_cols)
    remainder = length % num_cols
    if remainder == 0:
        remainder = num_cols

    grid_columns = [[] for _ in range(num_cols)]
    index = 0
    for row in range(num_rows):
        cols_in_this_row = num_cols if row < num_rows - 1 or remainder == num_cols else remainder
        for col in range(cols_in_this_row):
            grid_columns[col].append(text[index])
            index += 1

    column_by_rank = [None] * num_cols
    for col_index, rank in enumerate(order):
        column_by_rank[rank] = col_index

    cipher_text = ""
    for col_index in column_by_rank:
        cipher_text += "".join(grid_columns[col_index])

    return cipher_text

def single_round_decrypt(cipher_text: str, keyword: str) -> str:
    validate_printable_text(cipher_text)

    num_cols = len(keyword)
    order = build_column_order(keyword)
    length = len(cipher_text)

    num_rows = math.ceil(length / num_cols)
    remainder = length % num_cols
    if remainder == 0:
        remainder = num_cols

    column_lengths = [num_rows if col < remainder else num_rows - 1 for col in range(num_cols)]

    column_by_rank = [None] * num_cols
    for col_index, rank in enumerate(order):
        column_by_rank[rank] = col_index

    grid_columns = [None] * num_cols
    position = 0
    for col_index in column_by_rank:
        col_len = column_lengths[col_index]
        grid_columns[col_index] = list(cipher_text[position:position + col_len])
        position += col_len

    plain_text = []
    column_pointers = [0] * num_cols
    for row in range(num_rows):
        cols_in_this_row = num_cols if row < num_rows - 1 or remainder == num_cols else remainder
        for col in range(cols_in_this_row):
            plain_text.append(grid_columns[col][column_pointers[col]])
            column_pointers[col] += 1

    return "".join(plain_text)

### 4. Generate a Random Key (List of Round Keywords)

In [4]:
def generate_random_key(num_rounds: int = 2, min_length: int = 4, max_length: int = 10) -> list:
    # one random keyword per round, each round can have a different length
    keywords = []
    for _ in range(num_rounds):
        length = random.randint(min_length, max_length)
        keyword = "".join(random.choice(string.ascii_uppercase) for _ in range(length))
        keywords.append(keyword)
    return keywords

### 5. Multi-Round Encryption

In [5]:
def encrypt(text: str, keywords: list) -> str:
    if len(keywords) < 1:
        raise ValueError("'keywords' must contain at least one keyword.")

    result = text
    for keyword in keywords:
        result = single_round_encrypt(result, keyword)

    return result

### 6. Multi-Round Decryption

In [6]:
def decrypt(cipher_text: str, keywords: list) -> str:
    if len(keywords) < 1:
        raise ValueError("'keywords' must contain at least one keyword.")

    result = cipher_text
    # undo the rounds in reverse order: last keyword used first during decryption
    for keyword in reversed(keywords):
        result = single_round_decrypt(result, keyword)

    return result

### 7. Verify the Hand Worked Example in Code

In [7]:
hand_plaintext = "ATTACKATDAWN"
hand_keywords = ["HACK", "ZEBRA"]

round_one_output = single_round_encrypt(hand_plaintext, hand_keywords[0])
print(f"After Round 1 (keyword HACK): {round_one_output}  (should match TKATAWACDATN)")

final_cipher = encrypt(hand_plaintext, hand_keywords)
print(f"After Round 2 (keyword ZEBRA): {final_cipher}  (should match AAACKANTDTWT)")

hand_decrypted = decrypt(final_cipher, hand_keywords)
print(f"Decrypted back: {hand_decrypted}")

After Round 1 (keyword HACK): TKATAWACDATN  (should match TKATAWACDATN)
After Round 2 (keyword ZEBRA): AAACKANTDTWT  (should match AAACKANTDTWT)
Decrypted back: ATTACKATDAWN


### 8. Example usage

In [12]:
keys = generate_random_key(num_rounds=3)
print(f"Generated Random Key (one keyword per round): {keys}")

Generated Random Key (one keyword per round): ['ONBNENMOW', 'IOZO', 'TCTGVGD']


In [9]:
plaintext = """TOP secret Massage! Agent 101, visit Area 51 (37d14'0\"N 115d48'30\"W)."""
print(f"Original Plain Text: {plaintext}")

cipher_text = encrypt(plaintext, keys)
print("Encrypted:", cipher_text)

decrypted_text = decrypt(cipher_text, keys)
print("Decrypted:", decrypted_text)

match = plaintext == decrypted_text
print(f"Verification Match:{match}")

Original Plain Text: TOP secret Massage! Agent 101, visit Area 51 (37d14'0"N 115d48'30"W).
Encrypted:  ( e dT,st1v" 01s14!71r "a'Otr4e.Nan0Mtd) Aae s1i80c 5g 'eP5Ws133geiA
Decrypted: TOP secret Massage! Agent 101, visit Area 51 (37d14'0"N 115d48'30"W).
Verification Match:True


### 9. Comparing Single Round vs Multiple Rounds

This cell encrypts the same message using only round 1, then using all rounds, so you can see how much more scrambled the text becomes as more rounds are added.

In [10]:
for round_count in range(1, len(keys) + 1):
    partial_keys = keys[:round_count]
    partial_cipher = encrypt(plaintext, partial_keys)
    partial_decrypted = decrypt(partial_cipher, partial_keys)
    print(f"Rounds={round_count} Keywords={partial_keys}")
    print(f"Cipher: {partial_cipher}")
    print(f"Match:  {partial_decrypted == plaintext}\n")

Rounds=1 Keywords=['AQRRLOFSR']
Cipher: Tt!0 (N3cats5'4sseva15)esni 4d.O  1A3 0PMA,r71" ag ed1Wee1t "'rg i108
Match:  True

Rounds=2 Keywords=['AQRRLOFSR', 'NMPLEGY']
Cipher:  svn P"1"0(5ai M W'80tesO01d 1tcs)d3r 1 T3454A,geg!ase. 7etiN'1 1Aaer
Match:  True

Rounds=3 Keywords=['AQRRLOFSR', 'NMPLEGY', 'EWCLAX']
Cipher:  ( e dT,st1v" 01s14!71r "a'Otr4e.Nan0Mtd) Aae s1i80c 5g 'eP5Ws133geiA
Match:  True

